# Prepare Data — NeMo Manifests & Ground-Truth RTTMs

This notebook:
1. Matches each audio file to its transcript PDF (via `transcripts.csv`)
2. Converts timestamps (`HH:MM:SS:CS`) to seconds
3. Generates **ground-truth RTTM** files from the transcripts
4. Generates a **NeMo manifest JSON** file ready for `run.sh`

**Naming convention:**  
`T-XXXX-L-NAME.wav` ↔ `T-XXXX-NAME-Transkr-Video.pdf`  
i.e. remove `-L-` from the audio stem.

In [ ]:
import json
import wave
import pandas as pd
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
BASE        = Path(".")                          # project root
AUDIO_DIR   = BASE / "audio"
TRANS_CSV   = BASE / "transcripts.csv"
GT_RTTM_DIR = BASE / "ground_truth" / "rttm"    # output: GT RTTM files
MANIFEST_OUT = BASE / "nemo-multistage-classroom-diarization" / "manifests" / "my_data.json"

GT_RTTM_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(TRANS_CSV)
print(f"Loaded {len(df)} utterances from {df['filename'].nunique()} transcripts")

In [ ]:
def ts_to_sec(ts: str) -> float:
    """Convert HH:MM:SS:CS (centiseconds) to seconds."""
    h, m, s, cs = ts.split(":")
    return int(h) * 3600 + int(m) * 60 + int(s) + int(cs) / 100

def audio_duration(wav_path: Path) -> float:
    with wave.open(str(wav_path)) as w:
        return w.getnframes() / w.getframerate()

def audio_to_pdf_name(audio_stem: str) -> str:
    """
    T-1101-L-Lek1  ->  T-1101-Lek1-Transkr-Video.pdf
    T-1103-L-Tutor-1-1  ->  T-1103-Tutor-1-1-Transkr-Video.pdf
    """
    parts = audio_stem.split("-", 3)   # ['T', '1101', 'L', 'Lek1'] or ['T', '1101', 'L', 'Tutor-1-1']
    return f"{parts[0]}-{parts[1]}-{parts[3]}-Transkr-Video.pdf"

In [ ]:
# ── Match audio files to transcripts ─────────────────────────────────────────
matched = []
unmatched = []

for wav in sorted(AUDIO_DIR.glob("*.wav")):
    pdf_name = audio_to_pdf_name(wav.stem)
    subset   = df[df["filename"] == pdf_name]
    if subset.empty:
        unmatched.append(wav.name)
    else:
        matched.append((wav, pdf_name, subset.reset_index(drop=True)))

print(f"Matched:   {len(matched)} files")
print(f"Unmatched: {len(unmatched)} files — {unmatched}")

In [ ]:
# ── Generate ground-truth RTTM files ─────────────────────────────────────────
#
# RTTM format:
#   SPEAKER <file_id> 1 <start> <duration> <NA> <NA> <speaker> <NA> <NA>
#
# Duration = next_utterance_start - current_start
# For the last utterance: audio_end - last_start (capped at 10s minimum)

# Speaker codes that mean "silence / non-speech" — skip these in RTTM
NON_SPEECH = {"O"}   # O = ambient sound, not a speaker

rttm_paths = {}

for wav, pdf_name, trans in matched:
    file_id  = wav.stem                       # e.g. T-1101-L-Tutor-1-1
    dur_audio = audio_duration(wav)
    rttm_out  = GT_RTTM_DIR / f"{file_id}.rttm"
    rttm_paths[wav] = rttm_out

    # Convert all timestamps once
    starts = trans["timestamp"].apply(ts_to_sec).tolist()

    lines = []
    for i, row in trans.iterrows():
        speaker = str(row["speaker"]) if pd.notna(row["speaker"]) else ""
        if not speaker or speaker in NON_SPEECH:
            continue

        start = starts[i]
        # duration = gap until next utterance (or until audio ends)
        end   = starts[i + 1] if i + 1 < len(starts) else dur_audio
        dur   = max(0.01, end - start)          # never negative

        lines.append(
            f"SPEAKER {file_id} 1 {start:.3f} {dur:.3f} <NA> <NA> {speaker} <NA> <NA>"
        )

    rttm_out.write_text("\n".join(lines) + "\n")
    print(f"{file_id}.rttm  —  {len(lines)} segments written")

In [ ]:
# ── Inspect the first RTTM ───────────────────────────────────────────────────
first_rttm = next(iter(rttm_paths.values()))
print(first_rttm.name)
lines = first_rttm.read_text().splitlines()
for l in lines[:10]:
    print(l)

In [ ]:
# ── Count speakers per recording (for NeMo manifest num_speakers) ────────────
#
# NeMo clusters into exactly `num_speakers` groups.
# Here we count distinct speaker LABELS per recording.
# Note: 'S', 'SN', 'Ss' are all student-side roles — they get merged
# into a single 'STU' group for a teacher/student 2-class distinction.

STUDENT_CODES = {"S", "SN", "Ss", "SS", "S?", "NS", "E"}
TEACHER_CODES = {"T", "L", "LN", "LS"}

speaker_counts = {}
for wav, pdf_name, trans in matched:
    codes = set(str(s) for s in trans["speaker"].dropna().unique())
    has_teacher = bool(codes & TEACHER_CODES)
    has_student = bool(codes & STUDENT_CODES)
    n = int(has_teacher) + int(has_student)
    speaker_counts[wav] = max(n, 2)   # at least 2
    print(f"{wav.stem:35s}  codes={sorted(codes)}  num_speakers={speaker_counts[wav]}")

In [ ]:
# ── Generate NeMo manifest JSON ───────────────────────────────────────────────
#
# One JSON object per line (JSON-Lines format).
# Paths must be absolute so NeMo can find them regardless of CWD.

manifest_lines = []

for wav, pdf_name, trans in matched:
    dur = audio_duration(wav)
    entry = {
        "audio_filepath": str(wav.resolve()),
        "offset":         0,
        "duration":       round(dur, 3),
        "label":          "infer",
        "text":           "-",
        "num_speakers":   speaker_counts[wav],
        "rttm_filepath":  str(rttm_paths[wav].resolve()),
        "uem_filepath":   None,
        "ctm_filepath":   None,
    }
    manifest_lines.append(json.dumps(entry, ensure_ascii=False))

MANIFEST_OUT.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_OUT.write_text("\n".join(manifest_lines) + "\n")
print(f"Manifest written to: {MANIFEST_OUT}")
print(f"Entries: {len(manifest_lines)}")
print()
for line in manifest_lines:
    e = json.loads(line)
    print(f"  {Path(e['audio_filepath']).name}  dur={e['duration']}s  n_spk={e['num_speakers']}")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
total_hours = sum(audio_duration(w) for w, _, _ in matched) / 3600
total_utterances = sum(len(t) for _, _, t in matched)

print("=" * 55)
print(f"  Files ready:      {len(matched)}")
print(f"  Total audio:      {total_hours:.2f} hours")
print(f"  GT utterances:    {total_utterances}")
print(f"  GT RTTM dir:      {GT_RTTM_DIR.resolve()}")
print(f"  Manifest:         {MANIFEST_OUT.resolve()}")
print("=" * 55)
print()
print("Next step: run the diarization pipeline")
print("  cd nemo-multistage-classroom-diarization")
print("  # Edit run.sh: set MANIFEST_FILE=./manifests/my_data.json")
print("  ./run.sh")